<a href="https://colab.research.google.com/github/farhadhsn8/MLP/blob/master/implementationDynamicMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Attribute Information:

1. sepal length in cm
2. sepal width in cm
3. petal length in cm
4. petal width in cm

## class:
1. Iris Setosa
2. Iris Versicolour
3. Iris Virginica

In [1]:
import matplotlib.pyplot as plt
from sklearn import datasets
import random
import numpy as np
import math
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [220]:
class MLP:


  def __init__(self , training_features , training_labels , parameters):
    self.parameters = parameters
    self.training_features = training_features
    self.training_labels = training_labels
    self.sliding_head = 0
    self.learning_rate = self.parameters['LEARNING_RATE']
    self.num_layers = len(self.parameters['CODE_OF_ACTIVATION_FUNCTIONS'])
    self.layers =  np.empty(self.num_layers,dtype=Layer)
    self.build_layers()
    
    

  def build_layers(self):
    for i in range(self.num_layers):
      self.layers[i] = Layer(i , self)

  def train(self):
    return self.predict_row(self.current_feature_row())


  def predict_row(self, X):
    self.reset_caches()
    return self.layers[self.num_layers - 1].calculateLayerOutput(X)


  def current_feature_row(self):
    return self.training_features[self.sliding_head]

  def current_label_row(self):
    return self.training_labels[self.sliding_head]


  def backpropagate(self):
    for layer in self.layers[:0:-1]:
      layer.update_weights( False)
    for layer in self.layers[:0:-1]:
      layer.update_weights( True)
    self.reset_caches()
    

  
  def train(self, epoch=1):
    
    for i in range(epoch):
      # printProgressBar(i, epoch, prefix = 'Progress:', suffix = 'Complete', length = 50)
      self.sliding_head =0
      for j in range(self.training_features.shape[0]):
        self.backpropagate()
        self.sliding_head +=1
      # printProgressBar(i + 1, epoch, prefix = 'Progress:', suffix = 'Complete', length = 50)

  def reset_caches(self):
    for i in range(self.num_layers):
      self.layers[i].resetOutput()
      for j in range(self.layers[i].num_neurons):
        self.layers[i].neurons[j].resetDelta()




  def clearAll(self):
      for i in range(self.num_layers):
        self.layers[i].resetOutput()
        for j in range(self.layers[i].num_neurons):
          self.layers[i].neurons[j].resetDelta()
          for k in range(self.layers[i].neurons[j].num_inputs):
            self.layers[i].neurons[j].input_branches[k].reset_weight()
            print(self.layers[i].neurons[j].input_branches[k].w)
          

      

  



    



#--------------------------------------------------------------------------


class Layer:

  def __init__(self,layer_index , MLP):
    self.MLP = MLP
    self.layer_index = layer_index
    self.num_neurons = self.setNumberOfPerceptrons()
    self.activation = ActivityFunction(self)
    self.neurons =  np.empty(self.num_neurons,dtype=Perceptron)
    self.neurons = self.build_neurons()
    self.output = np.full((self.num_neurons), math.inf)


  def resetOutput(self):
    self.output = np.full((self.num_neurons), math.inf)


  def setNumberOfPerceptrons(self):
    if(self.layer_index == 0 ):
      return  self.MLP.training_features.shape[1]
    if(self.layer_index == self.MLP.num_layers - 1 ):
      return  self.MLP.training_labels.shape[1]
    return self.MLP.parameters['NUMBER_OF_PERCEPTRONS_FOR_HIDDEN_LAYERS'][self.layer_index-1]

  def build_neurons(self):
    neurons =  np.empty(self.num_neurons,dtype=Perceptron) 
    for i in range( self.num_neurons ):
      neurons[i] = Perceptron( i , self)
    return neurons

  def getPreviousLayer(self):
    return self.layer_index != 0 and self.MLP.layers[self.layer_index - 1 ] or -1

  
  def getNextLayer(self):
    return self.layer_index != self.MLP.num_layers - 1 \
     and self.MLP.layers[self.layer_index + 1 ] or -1


  def calculateLayerOutput(self,X):     # receive Vector   # return Vector

    if ((any(self.output==math.inf))==False):
      return self.output
    if(self.layer_index==0):
      X = X
      return X
    else:
      X = self.getPreviousLayer().calculateLayerOutput(X)
    output =  np.empty(self.num_neurons)
    for i in range(self.num_neurons):
      if(self.layer_index == 0 ):
        output[i] = X[i]
      else:
        output[i] = self.neurons[i].forward(X)
    self.output = output
    return self.output

  def derivative(self,net):
    return self.activation.calculateDerivative(net)

  def update_weights(self, hardUpdate = False):
    for perceptron in self.neurons:
      perceptron.update_weights(hardUpdate)
    

  



#--------------------------------------------------------------------------





class ActivityFunction:
  
  def __init__(self,layer):
    self.layer = layer
    self.functionType = self.layer.MLP.parameters['CODE_OF_ACTIVATION_FUNCTIONS'][self.layer.layer_index]
  
  def apply(self,x):
    if (self.functionType == 1) :
      return self.sigmoid(x)
    if (self.functionType == 2) :
      return self.tanh(x)
    if (self.functionType == 3) :
      return self.ReLU(x)
    if (self.functionType == 4) :
      return self.linear(x)

  def sigmoid(self, x):
    return 1 / (1 + math.exp(-x))

  def tanh(self , x):
    t=(math.exp(x)-math.exp(-x))/(math.exp(x)+math.exp(-x))
    return t

  def ReLU(self ,x):
    return max(0.0,x)

  def linear(self , x):
    return x

  def calculateDerivative(self , net):
    if (self.functionType == 1) :
      sig = self.sigmoid(net)
      return (1-sig)*sig
    if (self.functionType == 2) :
      return 1 - self.tanh(net)**2
    if (self.functionType == 3) :
      if(net<0):
        return 0
      return 1
    if (self.functionType == 4) :
      return 1



#--------------------------------------------------------------------------


class Perceptron:

  

  def __init__(self , perceptron_index , layer ):   # [layer_index  ,  perceptron] 
    self.bias = 0  # 0 or 1
    self.perceptron_index = perceptron_index
    self.layer = layer
    self.num_inputs  =  self.getNumberOfInputs()
    self.input_branches =  np.empty(self.num_inputs,dtype=Layer)
    self.build_inputs()
    self.delta = math.inf

  def resetDelta(self):
    self.delta = math.inf

  def build_inputs(self):
    for i in range(self.num_inputs):
      self.input_branches[i] = InputBranch(self , i)

  def getNumberOfInputs(self):
    if(self.layer.layer_index == 0 ):
      return  1
    return self.layer.getPreviousLayer().num_neurons + self.bias

  def forward(self , X):
        net = self.net_output(X)
        return self.layer.activation.apply(net)

        
  def net_output(self , X):    # X is input feature vector
        y=0
        # DONT FORGET BAIAS
        X = np.concatenate((X, [self.bias]), axis=None)
        for i in range(self.num_inputs):
          y += self.input_branches[i].branch_output(X[i])
        return y


  def getDelta(self):
    if(self.delta != math.inf):
      return self.delta
    desiredOutput=0
    if(self.layer.layer_index==self.layer.MLP.num_layers - 1):
      desiredOutput = self.layer.MLP.current_label_row()[self.perceptron_index]
    X = (self.layer.layer_index == 0)  and self.layer.MLP.current_feature_row() or self.layer.getPreviousLayer().calculateLayerOutput(self.layer.MLP.current_feature_row())
    self.delta =  self.calculateDelta(X ,desiredOutput)
    return self.delta
    # print(self.layer.layer_index, self.perceptron_index,self.delta)

  def calculateDelta(self,X , desiredOutput):  # X is input vector 
    net = self.net_output(X)
    if(self.layer.layer_index == self.layer.MLP.num_layers - 1):     # perceptron in output layer
      return self.layer.derivative(net) * ( desiredOutput - self.forward(X))
    else:       # perceptron in hidden layer
      sigma = 0
      # layerOutput = self.layer.calculateLayerOutput(self.layer.MLP.current_feature_row())
      for perceptron in self.layer.getNextLayer().neurons:
        sigma += (perceptron.input_branches[self.perceptron_index].w * perceptron.getDelta()) 
      return self.layer.derivative(net) * sigma

  
  def update_weights(self,hardUpdate = False):
    for inputBranch in self.input_branches:
      hardUpdate and inputBranch.apply_w_new() or inputBranch.updatew_new()


    


  
#--------------------------------------------------------------------------

class InputBranch:
  
  def __init__(self , perceptron, inputNumber):
    self.inputNumber = inputNumber
    self.perceptron = perceptron
    self.reset_weight()
    self.w_new = self.w

  def reset_weight(self):
    if(self.perceptron.layer.layer_index == 0):
      self.w =  1
    self.w = random.uniform(0,1) #00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

  def branch_output(self , x):
    return self.w * x 

  

  def updatew_new(self):
    learning_rate = self.perceptron.layer.MLP.learning_rate
    yi = np.concatenate((self.perceptron.layer.getPreviousLayer().calculateLayerOutput(self.perceptron.layer.MLP.current_feature_row()), [self.perceptron.bias]), axis=None)[self.inputNumber]
    self.w_new =self.w +  learning_rate * self.perceptron.getDelta() * yi 

  def apply_w_new(self):
    self.w = self.w_new





In [671]:
iris = datasets.load_iris()
features = iris.data  
target = pd.get_dummies(iris.target).to_numpy()
features.shape   # (150, 4)
target.shape

dataset = np.hstack(( features,target,np.reshape(iris.target,(-1,1))))
#---------------shuffle---------------------
from sklearn.utils import shuffle
dataset=shuffle(dataset)

#-------------test & train ---------------
train=dataset[0:120,:]    
test=dataset[120:,:]  
test.shape                #(30, 7)
train.shape              # (120, 7)


# dataset[149]


(120, 8)

In [678]:
PARAMS = {
    # enter learning rate :0.1
    # enter code of function for layer0 =>[ 1.sigmoid  | 2.tanh  | 3.relu | 4.linear ] :4
    # enter number of Perceptrons for  layer 1 (start layer number from 0) : 2
    # enter code of function for layer1 =>[ 1.sigmoid  | 2.tanh  | 3.relu | 4.linear ] :4
    # enter code of function for layer2 =>[ 1.sigmoid  | 2.tanh  | 3.relu | 4.linear ] :4
    'LEARNING_RATE' : 0.01 ,
    'CODE_OF_ACTIVATION_FUNCTIONS' : [4,2,3] , #[ 1.sigmoid  | 2.tanh  | 3.relu | 4.linear ]
    'NUMBER_OF_PERCEPTRONS_FOR_HIDDEN_LAYERS' : [5]
  }

PARAMS['NUMBER_OF_PERCEPTRONS_FOR_HIDDEN_LAYERS']



[5]

In [679]:
IRIS_MLP = MLP(train[:,0:4] ,train[:,4:7] ,PARAMS )
# IRIS_MLP = MLP(train[:,0:4] ,5 * np.reshape(train[:,7],(-1,1,1)) ,PARAMS)
# IRIS_MLP = MLP(np.array([[1,1]]) ,np.array([[2,2]]) , PARAMS )
# IRIS_MLP = MLP(s1 ,s2, PARAMS )

In [680]:
# IRIS_MLP.clearAll()

In [681]:

IRIS_MLP.train(100)

In [683]:
s=0
k=0
for i in test: #test or train
  est = IRIS_MLP.predict_row(i[0:4])
  print(est , i[4:7])
  k+=int(np.argmax(est) == np.argmax(i[4:7]) )
  s+=1

  

k, s , str(k/s * 100)+'%'

 # # print([IRIS_MLP.predict_row(s1[i])[0] ,IRIS_MLP.predict_row(s1[i])[1]] , list(s2[i]))
 # # s+=int([int(IRIS_MLP.predict_row(s1[i])[0]>0) ,int(IRIS_MLP.predict_row(s1[i])[1]>0)] == list(s2[i]))
# print(IRIS_MLP.predict_row([1,1]) )

[0.99848164 0.17198087 0.        ] [1. 0. 0.]
[0.99823584 0.17201416 0.        ] [1. 0. 0.]
[0.         0.38055866 0.99072584] [0. 0. 1.]
[0.99604302 0.17218898 0.        ] [1. 0. 0.]
[0.99728197 0.17216161 0.        ] [1. 0. 0.]
[0.        0.3026091 0.       ] [0. 1. 0.]
[0.         0.33196034 0.17810801] [0. 1. 0.]
[0.         0.38240142 1.02152945] [0. 0. 1.]
[0.         0.33600394 0.24572055] [0. 1. 0.]
[0.98805872 0.17302123 0.        ] [1. 0. 0.]
[0.99898129 0.17227874 0.        ] [1. 0. 0.]
[0.         0.37917907 0.96763975] [0. 0. 1.]
[0.97976824 0.17394422 0.        ] [1. 0. 0.]
[0.         0.31213527 0.        ] [0. 1. 0.]
[0.99441637 0.17238797 0.        ] [1. 0. 0.]
[0.         0.36742499 0.77111898] [0. 0. 1.]
[0.         0.33139169 0.16921235] [0. 1. 0.]
[0.         0.38264939 1.0256649 ] [0. 0. 1.]
[0.         0.3705859  0.82401075] [0. 1. 0.]
[0.99665289 0.17220335 0.        ] [1. 0. 0.]
[0.         0.38167945 1.00944926] [0. 0. 1.]
[0.         0.38050263 0.98977212] [0

(29, 30, '96.66666666666667%')

In [684]:
for i in range(IRIS_MLP.num_layers):
  print('layer'+str(i))
  for j in range(IRIS_MLP.layers[i].num_neurons):
    print('percepron'+str(j))
    for k in range(IRIS_MLP.layers[i].neurons[j].num_inputs):
      print(IRIS_MLP.layers[i].neurons[j].input_branches[k].w)

layer0
percepron0
0.6745741915769821
percepron1
0.7228124777241237
percepron2
0.2240955526384436
percepron3
0.7604801556986462
layer1
percepron0
-0.2675212637673663
-0.7665263801452425
0.6370509415846145
1.1836632022822173
percepron1
0.4836541800763478
0.6836016974115433
0.1282189587269721
0.7420572263552115
percepron2
0.4805091716517456
0.7603537062079601
0.7408765129602126
0.6396585767737258
percepron3
0.3517275271994939
0.6309296244390415
0.7935077330435562
0.7092022370811348
percepron4
0.6145790530678699
0.060287918595537365
0.6798215817599988
0.46831036896129835
layer2
percepron0
-1.1161823602646326
-0.05089353945017402
-0.08852231696908709
0.387556356525203
-0.3612706469169865
percepron1
0.10699432331504279
0.45662882565550805
0.3094868202759435
-0.040245292057601134
-0.4474039319375628
percepron2
1.7890402397912282
0.3079120254786013
-0.5599813758330291
0.0028568307193348606
-0.4671549026249559
